# CyberCodeMini v0.3.0 — Google Colab Training Notebook

**Objective**: Supervised Fine-Tuning (SFT) of `Qwen/Qwen2.5-Coder-1.5B-Instruct` on the frozen **CyberCodeMini v0.3.0** dataset (900 training candidates, 90 validation candidates, 200 evaluation-only items).

### Hardware Requirement:
- Free **T4 GPU** (15 GB VRAM) or **A100 GPU** on Google Colab.
- Estimated Training Time: ~25–40 minutes on T4 (or ~8 minutes on A100).

## 1. Verify GPU Hardware & Environment

In [ ]:
!nvidia-smi

## 2. Install Dependencies & Clone Fresh Repository

In [ ]:
import os, shutil, sys

# 1. Install required PyPI packages
!pip install -q torch transformers peft datasets accelerate bitsandbytes trl pyyaml pydantic

# 2. Always reset working directory to /content
%cd /content

# 3. Wipe old repository folder to guarantee 100% fresh checkout without stale directories
if os.path.exists("/content/cybercode_v1"):
    shutil.rmtree("/content/cybercode_v1", ignore_errors=True)

# 4. Clone fresh repository
!git clone https://github.com/griffin6569/cybercode_v1.git
%cd /content/cybercode_v1

## 3. Verify Frozen Dataset Integrity (v0.3.0)

In [ ]:
import hashlib
import json
from pathlib import Path

train_path = Path("data/frozen/v0.3.0/training.jsonl")
val_path = Path("data/frozen/v0.3.0/validation.jsonl")
manifest_path = Path("data/frozen/v0.3.0/manifest.json")

assert train_path.exists(), f"Training set missing: {train_path}"
assert val_path.exists(), f"Validation set missing: {val_path}"

# Check example row counts
train_rows = [json.loads(line) for line in train_path.read_text(encoding="utf-8").splitlines() if line.strip()]
val_rows = [json.loads(line) for line in val_path.read_text(encoding="utf-8").splitlines() if line.strip()]

print(f"✓ Train Examples Loaded: {len(train_rows)}")
print(f"✓ Val Examples Loaded:   {len(val_rows)}")
assert len(train_rows) == 900, f"Expected 900 training examples, got {len(train_rows)}"
assert len(val_rows) == 90, f"Expected 90 validation examples, got {len(val_rows)}"

# Cross-platform SHA-256 line-ending normalized check
raw_bytes = train_path.read_bytes()
norm_bytes = raw_bytes.replace(b"\r\n", b"\n")
train_sha256 = hashlib.sha256(norm_bytes).hexdigest()
expected_lf_sha256 = "2adf679749e085b227a6f7e5545fe5a7d2cd8f13d197873cee929e21701ddb14"

print(f"✓ Normalized Train Dataset SHA-256: {train_sha256}")
assert train_sha256 == expected_lf_sha256, f"SHA-256 mismatch! Got {train_sha256}"
print("✓ Frozen Dataset Integrity Verification PASSED!")

## 4. Execute LoRA Fine-Tuning

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

model_id = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

# 4-bit Quantization Config for T4 VRAM efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Load datasets
dataset = load_dataset("json", data_files={
    "train": "data/frozen/v0.3.0/training.jsonl",
    "validation": "data/frozen/v0.3.0/validation.jsonl"
})

def format_instruction(example):
    text = ""
    messages = example.get("messages", [])
    for msg in messages:
        role = msg.get("role", "user")
        content = msg.get("content", "")
        if role == "system":
            text += f"<|im_start|>system\n{content}<|im_end|>\n"
        elif role == "user":
            text += f"<|im_start|>user\n{content}<|im_end|>\n"
        elif role == "assistant":
            text += f"<|im_start|>assistant\n{content}<|im_end|>\n"
    return {"text": text}

formatted_dataset = dataset.map(format_instruction)

training_args = TrainingArguments(
    output_dir="outputs/cybercodemini_lora_v0.3.0",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=3,
    save_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    save_total_limit=2,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["validation"],
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args,
)

print("Starting CyberCodeMini Training...")
trainer.train()

# Save final adapter
trainer.model.save_pretrained("outputs/cybercodemini_lora_v0.3.0/final_adapter")
tokenizer.save_pretrained("outputs/cybercodemini_lora_v0.3.0/final_adapter")
print("✓ Model training complete! Saved to outputs/cybercodemini_lora_v0.3.0/final_adapter")

## 5. Interactive Inference Verification

In [ ]:
prompt = "<|im_start|>system\nYou are CyberCodeMini, an expert cybersecurity AI coding assistant.<|im_end|>\n<|im_start|>user\nWrite a Python function to safely sanitize user inputs to prevent SQL injection in SQLite queries.<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.2, top_p=0.9)
response = tokenizer.decode(outputs[0], skip_special_tokens=False)
print("--- CyberCodeMini Output ---")
print(response)

## 6. Compress Adapter Artifacts for Download

In [ ]:
!zip -r cybercodemini_lora_v0.3.0_adapter.zip outputs/cybercodemini_lora_v0.3.0/final_adapter
from google.colab import files
files.download("cybercodemini_lora_v0.3.0_adapter.zip")